# Lesson 9 : Workflows

Up until now, we have built a single agent with language models, tools, and messages.<br>
This type of agent essentially performs to call LLMs or tools until it reaches to the goal - such as, [ReAct](https://tsmatz.wordpress.com/2023/03/07/react-with-openai-gpt-and-langchain/)-style pattern. (See the left side in below picture.) But various processing patterns (such as, workflow pattern, fan-out/fan-in pattern, etc) or the mixture of these patterns will be required in production.

![Flow patterns](./assets/flow_patterns.png)

> Note : Especially in complex tasks, cramming all the process into a single agent will result into poor performance.  
> By dividing the roles among multiple agents and collaborating these blocks, the context becomes clear, and accuracy is then expected to improve. (Also, manageability will be improved.)

Using built-in process builders (such as, ```SequentialBuilder```, ```ConcurrentBuilder```, ```GroupChatBuilder```, ```HandoffBuilder```, and ```MagenticBuilder```) in Microsoft Agent Framework, you can build various processing patterns to orchestrate your existing agents and custom code without having to build them from scratch.<br>
```WorkflowBuilder``` is a generic builder for flexible workflow graphs. (Other builders are built upon this generic builder.)

In this exercise, we explore fundamental workflow's capabilities in Microsoft Agent Framework using the following brief sequential flow.

1. Create a plan for the trip.
2. Revise the generated plan.

> Note : Learning a variety of multi-agent design patterns (orchestration patterns) is out of scope in this repository. (See [official document](https://learn.microsoft.com/en-us/azure/architecture/ai-ml/guide/ai-agent-design-patterns) for this topic.)

## 1. Build and run a simple workflow

In this example, we build a workflow with generic ```WorkflowBuilder``` class in order to understand how it works internally.

> Note : You can easily build this type of simple sequential flows with built-in ```SequentialBuilder``` class (instead of generic ```WorkflowBuilder```), but we don't use this high-level class for your learning purpose.

First we initialize the client as usual.

In [1]:
from dotenv import load_dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity.aio import AzureCliCredential

load_dotenv()

credential = AzureCliCredential()
client = FoundryChatClient(credential=credential)

Now let's build a workflow and run as follows.

In workflow on Agent Framework, first we instantiate ```Executor``` (including ```Agent```) as building blocks.  
In this example, we use the following 3 executor instances - two is ```AgentExecutor``` and one is generic ```Executor```.

- planning_agent : This is an executor by LLM agent to generate a plan.
- revise_agent : This is an executor by LLM agent to revise a plan.
- response_generator : This executor generates the final response and finalize the workflow. (The workflow instance becomes idle state after this executor is run.) In this example, it returns a list of all messages for the final response.

As you can see in below code, a generic ```Executor``` is processed in a class method (function) decorated by ```@handler```. (In a single ```Executor```, you can also define multiple handlers depending on arguments.)  
All executors are connected by workflow context (the argument ```ctx``` in this code), and you can pass result (output) to another executor by using context. (See code in the following ```generate_final_respose()``` handler.)

> Note : It is not recommended to split a simple task into multiple agents, because it might degrade performance.  
> Please avoid overusing multi-agents. (This is for demo purpose.)

> Note : In this example, I have defined the workflow with code as follows, but workflows can also be defined with declarative description, YAML.

In [2]:
from agent_framework import Agent, Message
from agent_framework import (
    WorkflowBuilder,
    Executor,
    AgentExecutorResponse,
    WorkflowContext,
    handler
)

class ResponseGenerator(Executor):
    @handler
    async def generate_final_respose(
        self,
        response: AgentExecutorResponse,
        ctx: WorkflowContext[list[Message]]
    ) -> None:
        # in this example, we send full conversation as a final result.
        await ctx.yield_output(list(response.full_conversation))

# instantiate executor
response_generator = ResponseGenerator(id="final_response_generator")
planning_agent = Agent(
    name="PlanningAgent",
    client=client,
    instructions="Your task is to help users plan and concisely summarize it in five bullet points.",
)
revise_agent = Agent(
    name="ReviseAgent",
    client=client,
    instructions="Your task is to review the plan you receive and to refine it further.",
)

# build workflow
workflow = (
    WorkflowBuilder(start_executor=planning_agent, output_from="all")
    .add_edge(planning_agent, revise_agent)
    .add_edge(revise_agent, response_generator)
    .build()
)

# run workflow
events = await workflow.run("Create a plan for my travel in Osaka.")

Now we output the returned result (all messages in this example) as follows.

In [3]:
from agent_framework import Role

outputs = events.get_outputs()
for o in outputs:
    for msg in o.messages:
        print("==========================")
        print(f"[{msg.author_name}]")
        print(msg.text)

[PlanningAgent]
- **Pick your dates, base, and passes:** Choose 3–5 days; stay around **Namba/Shinsaibashi** (food/nightlife) or **Umeda** (transport). Consider **ICOCA** card, **Osaka Metro Day Pass**, and (if doing day trips) **JR Kansai Area Pass**.  
- **Day 1 – Classic Osaka core:** **Osaka Castle & park** → **Kuromon Ichiba Market** lunch → **Dotonbori + Shinsaibashi** evening street food and canal walk.  
- **Day 2 – Modern city + views:** **Umeda Sky Building** or **Abeno Harukas** observatory → **Osaka Aquarium Kaiyukan** + Tempozan area → sunset/night views and dinner in **Tenma** or **Nakanoshima**.  
- **Day 3 – Theme park or culture day:** Choose **Universal Studios Japan** (full day; book timed-entry options early) *or* a slower day with **Sumiyoshi Taisha**, **Shitennoji**, and retro **Shinsekai** (Tsutenkaku) + kushikatsu.  
- **Day 4–5 – Easy day trips (pick 1–2):** **Kyoto** (Fushimi Inari/Arashiyama), **Nara** (park + Todaiji), **Kobe** (harbor + beef), or **Himeji**

## 2. Run workflow with streaming

You can also run workflow with streaming.  
Especially when you handle checkpointing (pause/resume), human-in-the-loop (HITL), etc, this might be useful. (These topics will be discussed later.)

In [4]:
from agent_framework import AgentExecutorResponse

async for event in workflow.run(
    "Create a plan for my travel in Osaka.",
    stream=True):
    if event.type == "output":
        print(f"WorkflowEvent(type={event.type}, data='{event.data.text}')")
    else:
        print(f"WorkflowEvent(type={event.type})")

WorkflowEvent(type=started)
WorkflowEvent(type=status)
WorkflowEvent(type=executor_invoked)
WorkflowEvent(type=output, data='')
WorkflowEvent(type=output, data='')
WorkflowEvent(type=output, data='')
WorkflowEvent(type=output, data='')
WorkflowEvent(type=output, data='-')
WorkflowEvent(type=output, data=' **')
WorkflowEvent(type=output, data='Set')
WorkflowEvent(type=output, data=' basics')
WorkflowEvent(type=output, data=' (')
WorkflowEvent(type=output, data='when')
WorkflowEvent(type=output, data='/')
WorkflowEvent(type=output, data='where')
WorkflowEvent(type=output, data='/how')
WorkflowEvent(type=output, data='):')
WorkflowEvent(type=output, data='**')
WorkflowEvent(type=output, data=' Choose')
WorkflowEvent(type=output, data=' ')
WorkflowEvent(type=output, data='3')
WorkflowEvent(type=output, data='–')
WorkflowEvent(type=output, data='5')
WorkflowEvent(type=output, data=' days')
WorkflowEvent(type=output, data=';')
WorkflowEvent(type=output, data=' stay')
WorkflowEvent(type=outpu

## 3. Checkpoint

When you run a long-running workflow, it might need to suspend and restart it later.  
With checkpoint in workflow on Agent Framework, you can save (serialize) the state in the storage, and restore (deserialize) the state to resume the suspended instance.

In this example, we explore checkpoint using ```InMemoryCheckpointStorage``` (non-persistent storage) for demo purpose.

Checkpoint is created after each super step iteration internally.  
In this example, therefore, we suspend workflow when the first ```superstep_completed``` event is captured. And we then restore the checkpoint and resume the workflow.

First we generate a workflow again, with checkpoint enabled.

> Note : When you need to restore custom data in your ```Executor```, please implement ```on_checkpoint_save()``` and ```on_checkpoint_restore()``` methods (override methods) in your custom executor class. (This example doesn't have such data.)

In [5]:
from agent_framework import InMemoryCheckpointStorage

# create checkpoint storage
checkpoint_storage = InMemoryCheckpointStorage()

# instantiate executor
planning_agent = Agent(
    name="PlanningAgent",
    client=client,
    instructions="Your task is to help users plan and concisely summarize it in five bullet points.",
)
revise_agent = Agent(
    name="ReviseAgent",
    client=client,
    instructions="Your task is to review the plan you receive and to refine it further.",
)
response_generator = ResponseGenerator(id="final_response_generator")

# build workflow
workflow_builder = (
    WorkflowBuilder(
        start_executor=planning_agent,
        output_from="all",
        checkpoint_storage=checkpoint_storage,
    )
    .add_edge(planning_agent, revise_agent)
    .add_edge(revise_agent, response_generator)
)

Now let's run and suspend workflow as follows.  
As I have mentioned above, we suspend workflow when the first ```superstep_completed``` event is captured.

**You may encounter a detach error (exception) when exiting the loop, but please ignore it.**

In [6]:
output = ""

workflow = workflow_builder.build()

async for event in workflow.run("Create a plan for my travel in Osaka.", stream=True):
    # we suspend the workflow when the first superstep completion is captured
    if event.type == "superstep_completed":
        break
    # buffer the result on each executor completion
    elif event.type == "executor_completed":
        for d in event.data:
            if isinstance(d, AgentExecutorResponse):
                for msg in d.agent_response.messages:
                    output += "==========================\n"
                    output += f"[{msg.author_name}]\n"
                    output += msg.text
                    output += "\n"

Let's get all checkpoints and show the number of all saved checkpoints.  
Here we have 2 checkpoints - the first checkpoint is initial checkpoint (which is saved if there are messages from initial execution) and the second is created after the first super step iteration.

In [7]:
checkpoints = await checkpoint_storage.list_checkpoints(workflow_name=workflow.name)
print(len(checkpoints))

2


Now let's restore state by using checkpoint and resume workflow as follows.

In [8]:
workflow = workflow_builder.build()

final_checkpoint = checkpoints[-1]
async for event in workflow.run(checkpoint_id=final_checkpoint.checkpoint_id, stream=True):
    if event.type == "executor_completed":
        for d in event.data:
            if isinstance(d, AgentExecutorResponse):
                for msg in d.agent_response.messages:
                    output += "==========================\n"
                    output += f"[{msg.author_name}]\n"
                    output += msg.text
                    output += "\n"

Let's see the completed result (output) of this workflow.  
In this example, the output is mostly completed during the first execution (the execution up to the first ```superstep_completed``` event), but in long-running workflows involving human-in-the-loop, the state up to the suspend is preserved and carried over when resumed.

In [9]:
print(output)

[PlanningAgent]
- **Set trip basics:** Choose dates/season, budget, travel style (food, shopping, culture, nightlife, theme parks), and where you’ll stay (Namba/Dotonbori for nightlife, Umeda for transit, Shin-Osaka for convenience).  
- **Build a simple 3–5 day itinerary:** Day 1 Dotonbori + Shinsaibashi; Day 2 Osaka Castle + museum/river cruise; Day 3 Universal Studios Japan; Day 4 day trip (Kyoto/Nara/Kobe); Day 5 Kuromon Market + Sumiyoshi Taisha + last-minute shopping.  
- **Plan transport efficiently:** Get an ICOCA card, map key subway lines, consider Osaka Amazing Pass (sightseeing-heavy days) and USJ timed-entry strategy; reserve Shinkansen seats if doing multiple day trips.  
- **Lock in key reservations:** Book hotel early, USJ tickets/Express Pass (if desired), popular restaurants (okonomiyaki/kaiseki), and any tours (food crawl, bar hopping, Osaka Bay cruise).  
- **Organize essentials & daily flow:** Make a shortlist of must-eats (takoyaki, okonomiyaki, kushikatsu), group